In [1]:
pip install huggingface_hub

In [1]:
from huggingface_hub import snapshot_download

# Download the Qwen3-0.6B model
model_id = "Qwen/Qwen3-0.6B"
local_dir = "./qwen3-0.6b"

try:
    path = snapshot_download(repo_id=model_id, local_dir=local_dir)
    print(f"Model downloaded successfully to: {path}")
except Exception as e:
    print(f"An error occurred: {e}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Model downloaded successfully to: /content/qwen3-0.6b


In [2]:
pip install datasets

In [3]:
import json
from datasets import load_dataset

# We will download and inspect the structures of the datasets to see how their fields map to our target structure.
try:
    ds1 = load_dataset("ndtran0101/writing9-ielts-essays", split="train")
    print("ndtran0101 dataset columns:", ds1.column_names)
    print("Example:", ds1[0])
except Exception as e:
    print("Error loading first dataset:", e)

try:
    ds2 = load_dataset("chillies/ielts-writing-task2-essays", split="train")
    print("\nchillies dataset columns:", ds2.column_names)
    print("Example:", ds2[0])
except Exception as e:
    print("Error loading second dataset:", e)


README.md:   0%|          | 0.00/3.65k [00:00<?, ?B/s]

data/train.jsonl: reconstructing file:   0%|          |  0.00B /  260MB            

data/train.jsonl: downloading bytes:           |  0.00B            

data/test.jsonl: reconstructing file:   0%|          |  0.00B /  111MB            

data/test.jsonl: downloading bytes:           |  0.00B            

band9_augmented.jsonl:   0%|          | 0.00/6.84M [00:00<?, ?B/s]

... (5 outputs left out)

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 8.28MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8049 [00:00<?, ? examples/s]


chillies dataset columns: ['essay_id', 'topic', 'question', 'essay_text', 'gpt_advices', 'gpt_positive_highlights', 'overall_band', 'task_achievement_band', 'coherence_cohesion_band', 'lexical_resource_band', 'grammatical_range_band']
Example: {'essay_id': '66abd675d091c90011b44e19', 'topic': 'Art', 'question': 'In many countries, people now wear western clothes such as suits and jeans rather than traditional clothing. Why is this the case? Is this a positive or negative development?', 'essay_text': 'In some nations, individuals tend to use Western-style attire instead of traditional dress. In my opinion, this a result of globalization and this change ends in reducing the importance of cultural heritage which is a huge demerit.\n\nNowadays, men and women are fond of fashionable items especially when it comes to clothing. This clearly states the effects of global change. In the past centuries, consumers were limited to clothes that they either made by themselves by knitting or buying f

In [4]:
# Prepare the transformation and saving logic
import json
import os

processed_data = []

# 1. Process ndtran0101/writing9-ielts-essays
# We map standard fields (prompt/essay -> user input, grades/feedback -> assistant assessment json)
try:
    for item in ds1:
        prompt = item.get('prompt', '') or item.get('question', '')
        essay = item.get('essay', '') or item.get('text', '')

        # Try parsing potential band score and components safely
        overall = item.get('band', 6.0) or item.get('score', 6.0)
        try:
            overall = float(overall)
        except:
            overall = 6.0

        assistant_content = {
            "overall_band": overall,
            "task_response": item.get('task_achievement', overall),
            "coherence_cohesion": item.get('coherence_cohesion', overall),
            "lexical_resource": item.get('lexical_resource', overall),
            "grammar": item.get('grammatical_range_accuracy', overall),
            "strengths": [],
            "weaknesses": [],
            "feedback": item.get('feedback', 'Good essay structure.')
        }

        message = {
            "messages": [
                {
                    "role": "system",
                    "content": "You are an IELTS Writing assessment assistant. Evaluate essays using IELTS Writing band descriptors."
                },
                {
                    "role": "user",
                    "content": f"Question: {prompt}\n\nEssay: {essay}"
                },
                {
                    "role": "assistant",
                    "content": json.dumps(assistant_content, ensure_ascii=False)
                }
            ]
        }
        processed_data.append(message)
    print(f"Successfully processed {len(ds1)} samples from ndtran0101/writing9-ielts-essays.")
except Exception as e:
    print("Error mapping ds1:", e)

# 2. Process chillies/ielts-writing-task2-essays
try:
    for item in ds2:
        prompt = item.get('prompt', '') or item.get('question', '')
        essay = item.get('essay', '') or item.get('text', '')
        overall = item.get('band', 6.0) or item.get('score', 6.0)
        try:
            overall = float(overall)
        except:
            overall = 6.0

        assistant_content = {
            "overall_band": overall,
            "task_response": overall,
            "coherence_cohesion": overall,
            "lexical_resource": overall,
            "grammar": overall,
            "strengths": [],
            "weaknesses": [],
            "feedback": item.get('feedback', '') or 'Evaluated based on general IELTS standards.'
        }

        message = {
            "messages": [
                {
                    "role": "system",
                    "content": "You are an IELTS Writing assessment assistant. Evaluate essays using IELTS Writing band descriptors."
                },
                {
                    "role": "user",
                    "content": f"Question: {prompt}\n\nEssay: {essay}"
                },
                {
                    "role": "assistant",
                    "content": json.dumps(assistant_content, ensure_ascii=False)
                }
            ]
        }
        processed_data.append(message)
    print(f"Successfully processed total {len(processed_data)} samples so far.")
except Exception as e:
    print("Error mapping ds2:", e)

# Save dataset to dataset.jsonl
output_file = "dataset.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for item in processed_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Finished building file: {output_file}")


Successfully processed 114504 samples from ndtran0101/writing9-ielts-essays.
Successfully processed total 122553 samples so far.
Finished building file: dataset.jsonl


In [36]:
pip install trl peft accelerate bitsandbytes

In [37]:
!pip install -U torchao

In [40]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig

# Path to the downloaded model and training dataset
model_id = "./qwen3-0.6b"
dataset_path = "dataset.jsonl"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Configure LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Load the dataset (keep the "messages" column as-is)
training_dataset = load_dataset("json", data_files=dataset_path, split="train")

# Load the base model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Configure SFTConfig with assistant_only_loss=True
training_args = SFTConfig(
    output_dir="./ielts-qwen-output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100,
    optim="paged_adamw_8bit",
    fp16=True,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    save_strategy="steps",
    save_steps=50,
    report_to="none",
    max_length=1024,
    assistant_only_loss=True,  # Replaces DataCollatorForCompletionOnlyLM
)

# SFTTrainer automatically applies the chat template to the "messages" column
trainer = SFTTrainer(
    model=model,
    train_dataset=training_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

# Start training
trainer.train()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Step,Training Loss
10,1.190368
20,0.020593
30,0.002583
40,0.000911
50,0.000098
60,0.000074
70,0.000067
80,0.000046
90,0.000042
100,0.000048


TrainOutput(global_step=100, training_loss=0.12148281401226996, metrics={'train_runtime': 269.8138, 'train_samples_per_second': 2.965, 'train_steps_per_second': 0.371, 'total_flos': 1106306826240000.0, 'train_loss': 0.12148281401226996, 'epoch': 0.006528054313411888})

In [41]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_path = "./qwen3-0.6b"
adapter_path = "./ielts-qwen-output/checkpoint-100"
merged_model_path = "./ielts-qwen-merged"

# Load the base model in FP16
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Load and merge the LoRA adapter
model = PeftModel.from_pretrained(base_model, adapter_path)
model = model.merge_and_unload()

# Save the merged model and tokenizer
model.save_pretrained(merged_model_path)
tokenizer = AutoTokenizer.from_pretrained(adapter_path)
tokenizer.save_pretrained(merged_model_path)

print("Merged model saved to:", merged_model_path)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: ./ielts-qwen-merged


In [43]:
import json

config_path = "./ielts-qwen-merged/tokenizer_config.json"

with open(config_path, "r") as f:
    config = json.load(f)

if isinstance(config.get("extra_special_tokens"), list):
    config.pop("extra_special_tokens")
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)
    print("Fixed tokenizer_config.json")

Fixed tokenizer_config.json


In [44]:
!python llama.cpp/convert_hf_to_gguf.py ./ielts-qwen-merged \
  --outfile ielts-qwen3-0.6b-q8_0.gguf \
  --outtype q8_0

INFO:hf-to-gguf:Loading model: ielts-qwen-merged
INFO:hf-to-gguf:Model architecture: Qwen3ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> Q8_0, shape = {1024, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {1024}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> Q8_0, shape = {3072, 1024}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> Q8_0, shape = {1024, 3072}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> Q8_0, shape = {1024, 3072}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {1024}
INFO:hf-to-gguf:blk.0.attn_k_norm.weight,  torch.float16 --> F32, shape = {128}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> Q8_0, shape = {1024, 1024}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch

In [42]:
!git clone https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements.txt

# Convert to Q8_0 GGUF (recommended for 0.6B models — fast, ~650MB, minimal quality loss)
!python llama.cpp/convert_hf_to_gguf.py ./ielts-qwen-merged \
  --outfile ielts-qwen3-0.6b-q8_0.gguf \
  --outtype q8_0

Cloning into 'llama.cpp'...
remote: Enumerating objects: 168176, done.
remote: Counting objects: 100% (502/502), done.
remote: Compressing objects: 100% (209/209), done.
remote: Total 168176 (delta 348), reused 364 (delta 293), pack-reused 167674 (from 2)
Receiving objects: 100% (168176/168176), 502.79 MiB | 26.07 MiB/s, done.
Resolving deltas: 100% (119250/119250), done.
Updating files: 100% (4621/4621), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is 

In [ ]:
from google.colab import files
files.download("ielts-qwen3-0.6b-q8_0.gguf")